# Finance — Hidden Periodic Exposures in Returns

Toy example: daily returns with a faint cyclical exposure (e.g., rebalancing, calendar effect).
We detect the period via FFT and illustrate the *explanatory* structure using `PeriodicState` QFT peaks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import rfft, rfftfreq
from quantum_hybrid_system import PeriodicState

rng = np.random.default_rng(1)
days = 600
t = np.arange(days)
# True period ~21 trading days (monthly-ish), plus noise
period_true = 21
ret = rng.normal(0, 0.01, size=days) + 0.002*np.sin(2*np.pi*t/period_true)

# FFT detection
yf = np.abs(rfft(ret - ret.mean()))
xf = rfftfreq(days, d=1.0)
k = np.argmax(yf[1:]) + 1
freq = xf[k]
period_est = 1/freq if freq>0 else np.nan
print("Estimated period ~", period_est)

plt.figure()
plt.plot(t, ret.cumsum())
plt.title("Cumulative returns (toy)")
plt.xlabel("day")
plt.ylabel("cum return")
plt.show()

# QFT explanatory picture
n = 10   # 1024 bins
r = max(2, min(int(round(period_est)), 64))
ps = PeriodicState(num_qubits=n, period=r)
samples = ps.measure(num_shots=4000, use_qft=True)

bins = 64
hist = np.zeros(bins, dtype=int)
N = 2**n
for s in samples:
    hist[(s * bins) // N] += 1

plt.figure()
plt.bar(np.arange(bins), hist)
plt.title(f"Analytic QFT sampling histogram (r≈{r})")
plt.xlabel("coarse frequency bin")
plt.ylabel("counts")
plt.show()